# lpr_cpp — LPR classifier Thrust device (GPU) check
Device eval parity + device TRAINING on a real T4. CPU-thrust: eval argmax 9/9 MATCH, training runs; nvcc -DUSE_CUDA compiles. **Runtime → GPU (T4)**, Run all.


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


In [ ]:
%cd /content
!rm -rf lpr_cpp
!git clone -q https://github.com/yomei-o/lpr_cpp.git
%cd /content/lpr_cpp


### 1. Device eval parity on GPU (ref weights + input are tracked)


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -diag-suppress 550 -Ipure/third_party pure/dnet_lpr_test.cpp -o dlpr_gpu
!./dlpr_gpu pure/ref/


Expect `argmax 9/9 MATCH` + `backend: GPU (CUDA)`.


### 2. Device TRAINING on GPU (9-head CE, synthetic — validates the loop is fast on GPU)


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -diag-suppress 550 -Xcompiler -fopenmp -Ipure/third_party pure/dtrain_lpr.cpp -o dtrain_gpu
!./dtrain_gpu 20 8 16 3e-4 pure/ref/


Expect the loss to compute and `backend: GPU (CUDA)`, much faster per step than the CPU-thrust build. (The 9-head cross-entropy is bridged to the host autograd; the conv/BN/backward run on device.) The YOLOX detector's own GPU check is under `yolox/colab/`.
